<a href="https://colab.research.google.com/github/IDispose/diffusion/blob/main/rf/examples/diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**RFdiffusion**
RFdiffusion is a method for structure generation, with or without conditional information (a motif, target etc). It can perform a whole range of protein design challenges as we have outlined in the RFdiffusion [manuscript](https://www.biorxiv.org/content/10.1101/2022.12.09.519842v2).

**<font color="red">NOTE:</font>** This notebook is in development, we are still working on adding all the options from the manuscript above.

For **instructions**, see end of Notebook.

See [diffusion_foldcond](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/main/rf/examples/diffusion_foldcond.ipynb) for fold conditioning functionality.

See [original version](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/main/rf/examples/diffusion_ori.ipynb) of this notebook (from 31Mar2023).

**<font color="red">Before you run anything:</font>** switch to a GPU runtime
(*Runtime → Change runtime type → T4 GPU*). The setup cell stops immediately if it
cannot see a GPU.

Colab's runtime is Python 3.13, but RFdiffusion depends on
[DGL](https://www.dgl.ai/), whose last release (2.4.0) only publishes wheels for
CPython 3.8–3.12 built against torch 2.4. Colab has no Python-version switch, so the
setup cell builds a self-contained **Python 3.11 environment** under
`/content/rfdiff-env` and runs `run_inference.py` with that interpreter. The kernel
itself only handles uploads, contig bookkeeping and the 3D views.

**<font color="red">Seeing `ModuleNotFoundError: No module named 'dgl'`?</font>** You are
running the old, pre-fix copy of this notebook (a copy saved in your Drive, or Colab's
cached copy of the GitHub file). This version never imports `dgl` into the Colab kernel.
Re-open
[the current notebook](https://colab.research.google.com/github/IDispose/diffusion/blob/main/rf/examples/diffusion.ipynb?flush_cache=true),
then *Runtime → Disconnect and delete runtime*, and run the setup cell again.

In [ ]:
%%time
#@title setup **RFdiffusion** (~8 min, **GPU runtime required**)
#@markdown Colab's hosted runtime is Python 3.13 / torch 2.11, but RFdiffusion depends on
#@markdown DGL, whose last release (2.4.0) only ships wheels for CPython 3.8-3.12 built
#@markdown against torch 2.4. There is no way to change the Python version of a Colab
#@markdown runtime, so this cell builds a **separate Python 3.11 environment** for
#@markdown RFdiffusion and runs `run_inference.py` under that interpreter. The notebook
#@markdown kernel stays on Colab's Python and only does file I/O and visualisation.
import os, time, signal, glob, subprocess
import sys, random, string, re

# Installs are quiet, but a quiet command that fails is useless -- capture the output and
# print the tail of it whenever the exit code is non-zero.
# NOTE: commands that background something must redirect it to a file inside the command
# (see the param download below), or this waits for that background job to close the pipe.
def sh(cmd, check=True, quiet=True):
  r = subprocess.run(cmd, shell=True, executable="/bin/bash", text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  out = (r.stdout or "").strip()
  if out and (r.returncode != 0 or not quiet):
    print("\n".join(out.split("\n")[-40:]))
  if r.returncode != 0 and check:
    raise RuntimeError(f"command failed (exit {r.returncode}): {cmd}")
  return r.returncode

# --- RFdiffusion is GPU-only in practice; fail fast instead of installing 3 GB of
# --- CUDA wheels into a runtime that cannot use them.
if sh("nvidia-smi -L > /dev/null 2>&1", check=False) != 0:
  raise RuntimeError("No GPU detected. RFdiffusion needs one: "
                     "Runtime > Change runtime type > T4 GPU, then re-run this cell.")

RFDIFF_ENV = "/content/rfdiff-env"
RFDIFF_PY  = f"{RFDIFF_ENV}/bin/python"

if not os.path.isdir("params"):
  sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq aria2 > /dev/null")
  sh("mkdir params")
  # send param download into background
  # 27Sep2026: https://files.ipd.uw.edu/krypton/{schedules.zip,ananas} now return 404.
  # schedules.zip was only a cache of the IGSO(3) tables, which RFdiffusion recomputes
  # (and caches in ./schedules) on first run; ananas is only used by symmetry="auto".
  sh("(\
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C params; \
  touch params/done.txt) > /content/params_download.log 2>&1 &")

if not os.path.isdir("RFdiffusion"):
  print("cloning RFdiffusion...")
  sh("git clone -q https://github.com/sokrypton/RFdiffusion.git")

if not os.path.isfile(RFDIFF_PY):
  print("building Python 3.11 environment for RFdiffusion (slowest step)...")
  sh("pip -q install uv")
  # --seed puts pip in the venv, so there is a fallback installer if uv chokes on a wheel.
  sh(f"uv venv -q --seed --python 3.11 {RFDIFF_ENV}")
  pip = f"VIRTUAL_ENV={RFDIFF_ENV} uv pip install -q"
  # the DGL wheel is 350 MB; uv's 30 s default HTTP timeout is not enough on a slow link
  os.environ["UV_HTTP_TIMEOUT"] = "600"

  # DGL first, with no deps: its metadata asks for an unpinned torch, which would pull
  # the latest release and break the ABI match its compiled extensions need.
  # Installed by direct wheel URL rather than through `-f .../repo.html`: data.dgl.ai is a
  # plain directory listing with one wheel per (python, torch, cuda) combo, and resolving
  # `dgl==2.4.0+cu124` out of it depends on the installer accepting both a local version
  # specifier and the legacy `manylinux1` tag.
  DGL_WHEEL = ("https://data.dgl.ai/wheels/torch-2.4/cu124/"
               "dgl-2.4.0%2Bcu124-cp311-cp311-manylinux1_x86_64.whl")
  if sh(f"{pip} --no-deps {DGL_WHEEL}", check=False) != 0:
    print("  uv could not install the DGL wheel; retrying with the venv's own pip...")
    sh(f"{RFDIFF_PY} -m pip install -q --no-deps {DGL_WHEEL}")

  # DGL bundles one prebuilt graphbolt library per torch version; pin torch to the
  # newest one present, otherwise `import dgl` dies with "Cannot find DGL C++ graphbolt".
  libs = glob.glob(f"{RFDIFF_ENV}/lib/python3.11/site-packages/dgl/graphbolt/libgraphbolt_pytorch_*.so")
  vers = sorted((os.path.basename(p)[len("libgraphbolt_pytorch_"):-3] for p in libs),
                key=lambda v: tuple(int(x) for x in v.split(".")))
  if not vers:
    raise RuntimeError("DGL wheel has no bundled graphbolt library -- did pip fall back "
                       "to an old dgl from PyPI instead of the cu124 wheel?")
  torch_ver = vers[-1]
  print(f"  pinning torch=={torch_ver} to match DGL's graphbolt build")
  sh(f"{pip} torch=={torch_ver} --index-url https://download.pytorch.org/whl/cu124")

  # DGL's own runtime imports (it reaches torchdata.datapipes, yaml, packaging, setuptools)
  sh(f"{pip} 'numpy<2' 'scipy<1.14' networkx requests tqdm psutil pandas pyyaml "
     f"packaging setuptools pydantic torchdata==0.8.0")

  # RFdiffusion + SE3Transformer
  sh(f"{pip} hydra-core omegaconf icecream pyrsistent decorator")
  # --no-deps on e3nn keeps it from resolving its unpinned `torch` requirement and
  # churning the pin above, so opt_einsum_fx's own dep on opt_einsum is named here.
  sh(f"{pip} --no-deps e3nn==0.5.5 opt_einsum_fx")
  sh(f"{pip} opt_einsum")
  sh(f"{pip} 'git+https://github.com/NVIDIA/dllogger#egg=dllogger'")
  sh(f"{pip} --no-deps ./RFdiffusion/env/SE3Transformer")

  # SE3Transformer calls dgl.ops.edge_softmax / e_dot_v / copy_e_sum, which only exist in
  # a real compiled DGL -- verify now rather than 10 minutes into a diffusion run.
  sh(f"DGLBACKEND=pytorch {RFDIFF_PY} -c \""
     "import torch, dgl;"
     "from dgl.ops import edge_softmax, e_dot_v, copy_e_sum;"
     "import se3_transformer;"
     "assert torch.cuda.is_available(), 'torch in the RFdiffusion env cannot see the GPU';"
     "print('  RFdiffusion env OK: python', __import__('sys').version.split()[0],"
     "'torch', torch.__version__, 'dgl', dgl.__version__)\"", quiet=False)

if not os.path.isdir("colabdesign"):
  print("installing ColabDesign...")
  sh("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
  # the ProteinMPNN/AlphaFold cell runs colabdesign/rf/designability_test.py by path, so
  # symlink the installed package here -- asking the interpreter where it landed instead
  # of globbing /usr/local/lib/python3.*/dist-packages, which silently makes a dangling
  # link whenever Colab's layout changes.
  where = subprocess.run([sys.executable, "-c",
                          "import colabdesign, os; print(os.path.dirname(colabdesign.__file__))"],
                         capture_output=True, text=True).stdout.strip()
  if not os.path.isdir(where):
    raise RuntimeError(f"could not locate the installed colabdesign package (got {where!r})")
  os.symlink(where, "colabdesign")

if not os.path.isdir("RFdiffusion/models"):
  print("waiting for RFdiffusion params...")
  sh("mkdir RFdiffusion/models")
  models = ["Base_ckpt.pt","Complex_base_ckpt.pt","Complex_beta_ckpt.pt"]
  for m in models:
    while os.path.isfile(f"{m}.aria2") or not os.path.isfile(m):
      time.sleep(5)
  sh(f"mv {' '.join(models)} RFdiffusion/models")

os.environ["DGLBACKEND"] = "pytorch"   # inherited by the run_inference.py subprocess

import json
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol

from colabdesign.rf.utils import get_ca
from colabdesign.rf.utils import fix_contigs, fix_partial_contigs, fix_pdb, sym_it
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

def parse_pdb(filename):
  '''Minimal stand-in for RFdiffusion's inference.utils.parse_pdb.

  Importing the real one would pull in dgl -> torch, which live in the Python 3.11
  env, not in this kernel. Everything downstream here (fix_contigs /
  fix_partial_contigs) only reads parsed_pdb["pdb_idx"], and this reproduces it
  exactly: (chain, resnum) per CA record, de-duplicated, in file order.
  '''
  pdb_idx = []
  for line in open(filename, "r"):
    if line[:4] == "ATOM" and line[12:16].strip() == "CA":
      idx = (line[21:22].strip(), int(line[22:26].strip()))
      if idx not in pdb_idx:
        pdb_idx.append(idx)
  return {"pdb_idx": pdb_idx}

def get_pdb(pdb_code=None):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    if not os.path.isfile(f"{pdb_code}.pdb1"):
      sh(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
      sh(f"gunzip {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"
  else:
    sh(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v3.pdb")
    return f"AF-{pdb_code}-F1-model_v3.pdb"

def run_ananas(pdb_str, path, sym=None):
  if not os.path.isfile("./ananas"):
    print("ERROR: the AnAnaS binary is no longer downloadable from files.ipd.uw.edu, so "
          "symmetry='auto' cannot run. Set symmetry='cyclic'/'dihedral' and `order` by hand, "
          "or drop an `ananas` binary next to this notebook.")
    return None, pdb_str

  pdb_filename = f"outputs/{path}/ananas_input.pdb"
  out_filename = f"outputs/{path}/ananas.json"
  with open(pdb_filename,"w") as handle:
    handle.write(pdb_str)

  cmd = f"./ananas {pdb_filename} -u -j {out_filename}"
  if sym is None: sh(cmd, check=False)
  else: sh(f"{cmd} {sym}", check=False)

  # parse results
  try:
    out = json.loads(open(out_filename,"r").read())
    results,AU = out[0], out[-1]["AU"]
    group = AU["group"]
    chains = AU["chain names"]
    rmsd = results["Average_RMSD"]
    print(f"AnAnaS detected {group} symmetry at RMSD:{rmsd:.3}")

    C = np.array(results['transforms'][0]['CENTER'])
    A = [np.array(t["AXIS"]) for t in results['transforms']]

    # apply symmetry and filter to the asymmetric unit
    new_lines = []
    for line in pdb_str.split("\n"):
      if line.startswith("ATOM"):
        chain = line[21:22]
        if chain in chains:
          x = np.array([float(line[i:(i+8)]) for i in [30,38,46]])
          if group[0] == "c":
            x = sym_it(x,C,A[0])
          if group[0] == "d":
            x = sym_it(x,C,A[1],A[0])
          coord_str = "".join(["{:8.3f}".format(a) for a in x])
          new_lines.append(line[:30]+coord_str+line[54:])
      else:
        new_lines.append(line)
    return results, "\n".join(new_lines)

  except Exception:
    return None, pdb_str

def run(command, steps, num_designs=1, visual="none", logfile=None):
  '''Launch `command` in the background and animate the trajectory it dumps to /dev/shm.

  Returns True on success. On failure it returns False instead of leaving the caller to
  trip over a missing output file.
  '''

  def run_command_and_get_pid(command):
    pid_file = '/dev/shm/pid'
    redirect = f"> {logfile} 2>&1" if logfile else ""
    os.system(f'nohup {command} {redirect} & echo $! > {pid_file}')
    with open(pid_file, 'r') as f:
      pid = int(f.read().strip())
    os.remove(pid_file)
    return pid
  def is_process_running(pid):
    try:
      os.kill(pid, 0)
    except OSError:
      return False
    else:
      return True

  run_output = widgets.Output()
  progress = widgets.FloatProgress(min=0, max=1, description='running', bar_style='info')
  display(widgets.VBox([progress, run_output]))

  # clear previous run
  for n in range(steps):
    if os.path.isfile(f"/dev/shm/{n}.pdb"):
      os.remove(f"/dev/shm/{n}.pdb")

  pid = run_command_and_get_pid(command)
  try:
    fail = False
    for _ in range(num_designs):

      # for each step check if output generated
      for n in range(steps):
        wait = True
        while wait and not fail:
          time.sleep(0.1)
          if os.path.isfile(f"/dev/shm/{n}.pdb"):
            pdb_str = open(f"/dev/shm/{n}.pdb").read()
            if pdb_str[-3:] == "TER":
              wait = False
            elif not is_process_running(pid):
              fail = True
          elif not is_process_running(pid):
            fail = True

        if fail:
          progress.bar_style = 'danger'
          progress.description = "failed"
          break

        else:
          progress.value = (n+1) / steps
          if visual != "none":
            with run_output:
              run_output.clear_output(wait=True)
              if visual == "image":
                xyz, bfact = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                fig = plt.figure()
                fig.set_dpi(100);fig.set_figwidth(6);fig.set_figheight(6)
                ax1 = fig.add_subplot(111);ax1.set_xticks([]);ax1.set_yticks([])
                plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                plt.show()
              if visual == "interactive":
                view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
                view.addModel(pdb_str,'pdb')
                view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
                view.zoomTo()
                view.show()
        if os.path.exists(f"/dev/shm/{n}.pdb"):
          os.remove(f"/dev/shm/{n}.pdb")
      if fail:
        break

    while is_process_running(pid):
      time.sleep(0.1)
    return not fail

  except KeyboardInterrupt:
    os.kill(pid, signal.SIGTERM)
    progress.bar_style = 'danger'
    progress.description = "stopped"
    return False

def run_diffusion(contigs, path, pdb=None, iterations=50,
                  symmetry="none", order=1, hotspot=None,
                  chains=None, add_potential=False, partial_T="auto",
                  num_designs=1, use_beta_model=False, visual="none"):

  full_path = f"outputs/{path}"
  os.makedirs(full_path, exist_ok=True)
  opts = [f"inference.output_prefix={full_path}",
          f"inference.num_designs={num_designs}"]

  if chains == "": chains = None

  # determine symmetry type
  if symmetry in ["auto","cyclic","dihedral"]:
    if symmetry == "auto":
      sym, copies = None, 1
    else:
      sym, copies = {"cyclic":(f"c{order}",order),
                     "dihedral":(f"d{order}",order*2)}[symmetry]
  else:
    symmetry = None
    sym, copies = None, 1

  # determine mode
  contigs = contigs.replace(","," ").replace(":"," ").split()
  is_fixed, is_free = False, False
  fixed_chains = []
  for contig in contigs:
    for x in contig.split("/"):
      a = x.split("-")[0]
      if a[0].isalpha():
        is_fixed = True
        if a[0] not in fixed_chains:
          fixed_chains.append(a[0])
      if a.isnumeric():
        is_free = True
  if len(contigs) == 0 or not is_free:
    mode = "partial"
  elif is_fixed:
    mode = "fixed"
  else:
    mode = "free"

  # fix input contigs
  if mode in ["partial","fixed"]:
    pdb_str = pdb_to_string(get_pdb(pdb), chains=chains)
    if symmetry == "auto":
      a, pdb_str = run_ananas(pdb_str, path)
      if a is None:
        print(f'ERROR: no symmetry detected')
        symmetry = None
        sym, copies = None, 1
      else:
        if a["group"][0] == "c":
          symmetry = "cyclic"
          sym, copies = a["group"], int(a["group"][1:])
        elif a["group"][0] == "d":
          symmetry = "dihedral"
          sym, copies = a["group"], 2 * int(a["group"][1:])
        else:
          print(f'ERROR: the detected symmetry ({a["group"]}) not currently supported')
          symmetry = None
          sym, copies = None, 1

    elif mode == "fixed":
      pdb_str = pdb_to_string(pdb_str, chains=fixed_chains)

    pdb_filename = f"{full_path}/input.pdb"
    with open(pdb_filename, "w") as handle:
      handle.write(pdb_str)

    parsed_pdb = parse_pdb(pdb_filename)
    opts.append(f"inference.input_pdb={pdb_filename}")
    if mode in ["partial"]:
      if partial_T == "auto":
        iterations = int(80 * (iterations / 200))
      else:
        iterations = int(partial_T)
      opts.append(f"diffuser.partial_T={iterations}")
      contigs = fix_partial_contigs(contigs, parsed_pdb)
    else:
      opts.append(f"diffuser.T={iterations}")
      contigs = fix_contigs(contigs, parsed_pdb)
  else:
    opts.append(f"diffuser.T={iterations}")
    parsed_pdb = None
    contigs = fix_contigs(contigs, parsed_pdb)

  if hotspot is not None and hotspot != "":
    hotspot = ",".join(hotspot.replace(","," ").split())
    opts.append(f"ppi.hotspot_res='[{hotspot}]'")

  # setup symmetry
  if sym is not None:
    sym_opts = ["--config-name symmetry", f"inference.symmetry={sym}"]
    if add_potential:
      sym_opts += ["'potentials.guiding_potentials=[\"type:olig_contacts,weight_intra:1,weight_inter:0.1\"]'",
                   "potentials.olig_intra_all=True","potentials.olig_inter_all=True",
                   "potentials.guide_scale=2","potentials.guide_decay=quadratic"]
    opts = sym_opts + opts
    contigs = sum([contigs] * copies,[])

  opts.append(f"'contigmap.contigs=[{' '.join(contigs)}]'")
  opts += ["inference.dump_pdb=True","inference.dump_pdb_path='/dev/shm'"]
  if use_beta_model:
    opts += ["inference.ckpt_override_path=./RFdiffusion/models/Complex_beta_ckpt.pt"]

  print("mode:", mode)
  print("output:", full_path)
  print("contigs:", contigs)

  opts_str = " ".join(opts)
  # run under the Python 3.11 interpreter, not the notebook kernel's Python
  cmd = f"{RFDIFF_PY} RFdiffusion/run_inference.py {opts_str}"
  print(cmd)

  # RUN
  logfile = f"{full_path}/run.log"
  ok = run(cmd, iterations, num_designs, visual=visual, logfile=logfile)
  if not ok:
    print(f"\nRFdiffusion failed. Last 40 lines of {logfile}:\n" + "-"*70)
    sh(f"tail -n 40 {logfile}", check=False, quiet=False)
    raise RuntimeError(f"RFdiffusion did not finish -- see {logfile}")

  # fix pdbs
  for n in range(num_designs):
    pdbs = [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
            f"outputs/traj/{path}_{n}_Xt-1_traj.pdb",
            f"{full_path}_{n}.pdb"]
    for pdb in pdbs:
      with open(pdb,"r") as handle: pdb_str = handle.read()
      with open(pdb,"w") as handle: handle.write(fix_pdb(pdb_str, contigs))

  return contigs, copies

In [ ]:
#@title check environment (optional)
#@markdown Prints what the notebook kernel has, and what the Python 3.11 RFdiffusion
#@markdown environment has. `run_inference.py` runs in the second one.
import subprocess, sys, textwrap

print("--- notebook kernel ---")
print("python:", sys.version.split()[0])
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "no GPU visible")

print("\n--- RFdiffusion env (%s) ---" % RFDIFF_PY)
probe = textwrap.dedent('''
    import sys, torch, dgl
    from dgl.ops import edge_softmax, e_dot_v, copy_e_sum
    import se3_transformer, e3nn, hydra
    print("python:", sys.version.split()[0])
    print("torch:", torch.__version__, "| cuda:", torch.version.cuda,
          "| available:", torch.cuda.is_available())
    print("dgl:", dgl.__version__, "| backend:", dgl.backend.backend_name)
    print("dgl.ops:", edge_softmax.__module__, e_dot_v.__module__, copy_e_sum.__module__)
    print("e3nn:", e3nn.__version__, "| se3-transformer + hydra import OK")
    if torch.cuda.is_available(): print("gpu:", torch.cuda.get_device_name(0))
''')
r = subprocess.run([RFDIFF_PY,"-c",probe], capture_output=True, text=True,
                   env={**os.environ, "DGLBACKEND":"pytorch"})
print(r.stdout or "probe failed -- re-run the setup cell")
if r.returncode != 0:
  print(r.stderr[-2000:])

In [ ]:
%%time
#@title run **RFdiffusion** to generate a backbone
name = "binder_sringeri" #@param {type:"string"}
contigs = "A:100-250" #@param {type:"string"}
pdb = "" #@param {type:"string"}
iterations = 25 #@param ["25", "50", "100", "150", "200"] {type:"raw"}
hotspot = "A200,A204,A350,A352,A354" #@param {type:"string"}
num_designs = 1 #@param ["1", "2", "4", "8", "16", "32"] {type:"raw"}
visual = "image" #@param ["none", "image", "interactive"]
#@markdown ---
#@markdown **symmetry** settings
#@markdown ---
symmetry = "none" #@param ["none", "auto", "cyclic", "dihedral"]
order = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"] {type:"raw"}
chains = "" #@param {type:"string"}
add_potential = True #@param {type:"boolean"}
#@markdown - `symmetry='auto'` enables automatic symmetry dectection with [AnAnaS](https://team.inria.fr/nano-d/software/ananas/).
#@markdown - `chains="A,B"` filter PDB input to these chains (may help auto-symm detector)
#@markdown - `add_potential` to discourage clashes between chains
#@markdown ---
#@markdown **advanced** settings
#@markdown ---
partial_T = "auto" # @param ["auto", "10", "20", "40", "60", "80"]
#@markdown - specify number of noising steps (only used for the partial diffusion protocol)
use_beta_model = False #@param {type:"boolean"}
#@markdown - if you are seeing lots of helices, switch to the "beta" params for a better SSE balance.

# determine where to save
path = name
while os.path.exists(f"outputs/{path}_0.pdb"):
  path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

flags = {"contigs":contigs,
         "pdb":pdb,
         "order":order,
         "iterations":iterations,
         "symmetry":symmetry,
         "hotspot":hotspot,
         "path":path,
         "chains":chains,
         "add_potential":add_potential,
         "num_designs":num_designs,
         "use_beta_model":use_beta_model,
         "visual":visual,
         "partial_T":partial_T}

for k,v in flags.items():
  if isinstance(v,str):
    flags[k] = v.replace("'","").replace('"','')

contigs, copies = run_diffusion(**flags)

In [ ]:
#@title Display 3D structure {run: "auto"}
animate = "none" #@param ["none", "movie", "interactive"]
color = "chain" #@param ["rainbow", "chain", "plddt"]
denoise = True
dpi = 100 #@param ["100", "200", "400"] {type:"raw"}
from colabdesign.shared.plot import pymol_color_list
from colabdesign.rf.utils import get_ca, get_Ls, make_animation
from string import ascii_uppercase,ascii_lowercase
alphabet_list = list(ascii_uppercase+ascii_lowercase)

def plot_pdb(num=0):
  if denoise:
    pdb_traj = f"outputs/traj/{path}_{num}_pX0_traj.pdb"
  else:
    pdb_traj = f"outputs/traj/{path}_{num}_Xt-1_traj.pdb"
  if animate in ["none","interactive"]:
    hbondCutoff = 4.0
    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    if animate == "interactive":
      pdb_str = open(pdb_traj,'r').read()
      view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    else:
      pdb = f"outputs/{path}_{num}.pdb"
      pdb_str = open(pdb,'r').read()
      view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    if color == "rainbow":
      view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
      for n,chain,c in zip(range(len(contigs)),
                              alphabet_list,
                              pymol_color_list):
          view.setStyle({'chain':chain},{'cartoon': {'color':c}})
    else:
      view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
    view.zoomTo()
    if animate == "interactive":
      view.animate({'loop': 'backAndForth'})
    view.show()
  else:
    Ls = get_Ls(contigs)
    xyz, bfact = get_ca(pdb_traj, get_bfact=True)
    xyz = xyz.reshape((-1,sum(Ls),3))[::-1]
    bfact = bfact.reshape((-1,sum(Ls)))[::-1]
    if color == "chain":
      display(HTML(make_animation(xyz, Ls=Ls, dpi=dpi, ref=-1)))
    elif color == "rainbow":
      display(HTML(make_animation(xyz, dpi=dpi, ref=-1)))
    else:
      display(HTML(make_animation(xyz, plddt=bfact*100, dpi=dpi, ref=-1)))


if num_designs > 1:
  output = widgets.Output()
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
      options=[(f'{k}',k) for k in range(num_designs)],
      value=0, description='design:',
  )
  dropdown.observe(on_change)
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

In [ ]:
%%time
#@title run **ProteinMPNN** to generate a sequence and **AlphaFold** to validate
#@markdown ProteinMPNN Settings
num_seqs = 8 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
mpnn_sampling_temp = 0.1 #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5", "1.0"] {type:"raw"}
rm_aa = "C" #@param {type:"string"}
use_solubleMPNN = False #@param {type:"boolean"}
#@markdown - `mpnn_sampling_temp` - control diversity of sampled sequences. (higher = more diverse).
#@markdown - `rm_aa='C'` - do not use [C]ysteines.
#@markdown - `use_solubleMPNN` - use weights trained only on soluble proteins. See [preprint](https://www.biorxiv.org/content/10.1101/2023.05.09.540044v2).
#@markdown
#@markdown AlphaFold Settings
initial_guess = False #@param {type:"boolean"}
#@markdown - soft initialization with desired coordinates, see [paper](https://www.nature.com/articles/s41467-023-38328-5).
num_recycles = 1 #@param ["0", "1", "2", "3", "6", "12"] {type:"raw"}
#@markdown - for **binder** design, we recommend `initial_guess=True num_recycles=3`
use_multimer = False #@param {type:"boolean"}
#@markdown - `use_multimer` - use AlphaFold Multimer v3 params for prediction.

if not os.path.isfile("params/done.txt"):
  print("downloading AlphaFold params...")
  while not os.path.isfile("params/done.txt"):
    time.sleep(5)

contigs_str = ":".join(contigs)
opts = [f"--pdb=outputs/{path}_0.pdb",
        f"--loc=outputs/{path}",
        f"--contig={contigs_str}",
        f"--copies={copies}",
        f"--num_seqs={num_seqs}",
        f"--num_recycles={num_recycles}",
        f"--rm_aa={rm_aa}",
        f"--mpnn_sampling_temp={mpnn_sampling_temp}",
        f"--num_designs={num_designs}"]
if initial_guess: opts.append("--initial_guess")
if use_multimer: opts.append("--use_multimer")
if use_solubleMPNN: opts.append("--use_soluble")
opts = ' '.join(opts)
!python colabdesign/rf/designability_test.py {opts}

In [ ]:
#@title Display best result
import py3Dmol
def plot_pdb(num = "best"):
  if num == "best":
    with open(f"outputs/{path}/best.pdb","r") as f:
      # REMARK 001 design {m} N {n} RMSD {rmsd}
      info = f.readline().strip('\n').split()
    num = info[3]
  hbondCutoff = 4.0
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
  pdb_str = open(f"outputs/{path}_{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  pdb_str = open(f"outputs/{path}/best_design{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})

  view.setStyle({"model":0},{'cartoon':{}}) #: {'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.setStyle({"model":1},{'cartoon':{'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.zoomTo()
  view.show()

if num_designs > 1:
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
    options=["best"] + [str(k) for k in range(num_designs)],
    value="best",
    description='design:',
  )
  dropdown.observe(on_change)
  output = widgets.Output()
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive,
#@markdown try disabling your adblocker and run this cell again.
#@markdown  If that fails click on the little folder icon to the
#@markdown  left, navigate to file: `name.result.zip`,
#@markdown  right-click and select \"Download\"
#@markdown (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).
!zip -r {path}.result.zip outputs/{path}* outputs/traj/{path}*
files.download(f"{path}.result.zip")

**Instructions**
---
---

Use `contigs` to define continious chains. Use a `:` to define multiple contigs and a `/` to define mutliple segments within a contig.
For example:

**unconditional**
- `contigs='100'` - diffuse **monomer** of length 100
- `contigs='50:100'` - diffuse **hetero-oligomer** of lengths 50 and 100
- `contigs='50'` `symmetry='cyclic'` `order=2` - make two copies of the defined contig(s) and add a symmetry constraint, for **homo-oligomeric** diffusion.

**binder design**
- `contigs='A:50'` `pdb='4N5T'` - diffuse a **binder** of length 50 to chain A of defined PDB.
- `contigs='E6-155:70-100'` `pdb='5KQV'` `hotspot='E64,E88,E96'` - diffuse a **binder** of length 70 to 100 (sampled randomly) to chain E and defined hotspot(s).

**motif scaffolding**
 - `contigs='40/A163-181/40'` `pdb='5TPN'`
 - `contigs='A3-30/36/A33-68'` `pdb='6MRR'` - diffuse a loop of length 36 between two segments of defined PDB ranges.

**partial diffusion**
- `contigs=''` `pdb='6MRR'` - noise all coordinates
- `contigs='A1-10'` `pdb='6MRR'` - keep first 10 positions fixed, noise the rest
- `contigs='A'` `pdb='1SSC'` - fix chain A, noise the rest

*hints and tips*
- `pdb=''` leave blank to get an upload prompt
- `contigs='50-100'` use dash to specify a range of lengths to sample from

**known limitations of this fork**
- `symmetry='auto'` needs the `ananas` binary, which `files.ipd.uw.edu/krypton/ananas`
  no longer serves (404). Set `symmetry='cyclic'`/`'dihedral'` and `order` by hand instead.
- `files.ipd.uw.edu/krypton/schedules.zip` is also gone, so the first diffusion run spends
  a few extra minutes computing the IGSO(3) schedules; they are then cached in `./schedules`.
- If a run fails, the RFdiffusion stderr is in `outputs/<name>/run.log`.